In [1]:
import numpy as np
import tensorflow as tf
import pandas as pd

In [2]:
ratings = pd.read_csv('datasets/ratings.csv')
movies = pd.read_csv('datasets/movies.csv', index_col='movieId')

In [3]:
loaded_data = np.load('recommender_weights.npz')

Y_pivot = ratings.pivot_table(index='movieId', columns='userId', values='rating').fillna(0)
Y = Y_pivot.to_numpy()
X = tf.Variable(loaded_data['X'], dtype=tf.float32)
W = tf.Variable(loaded_data['W'], dtype=tf.float32)
b = tf.Variable(loaded_data['b'], dtype=tf.float32)
movie_mean = tf.constant(loaded_data['movie_means'], dtype=tf.float32)

pred_Y = X @ tf.transpose(W) + b
pred_Y = pred_Y + movie_mean

In [4]:
userId = 123
userIdIdx = userId - 1
my_ratings = Y[:, userIdIdx]
rated_indices = np.where(my_ratings > 0)[0]
rated_movie_ids = Y_pivot.index[rated_indices]
print(f"Movies rated by User {userId}:")
display(movies.loc[rated_movie_ids])

Movies rated by User 123:


,title,genres
movieId,,
47,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
293,Léon: The Professional (a.k.a. The Professiona...,Action|Crime|Drama|Thriller
296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
318,"Shawshank Redemption, The (1994)",Crime|Drama
593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi
2329,American History X (1998),Crime|Drama
2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller


In [5]:
my_predictions = pred_Y[:, userIdIdx]

my_predictions = tf.where(
    my_ratings > 0,
    tf.constant(-1.0),
    my_predictions
)
top_10_indices = np.argsort(my_predictions)[::-1][:10]
top_10_movie_ids = Y_pivot.index[top_10_indices]

print(f"\nUser {userId} may like the following movies:")
display(movies.loc[top_10_movie_ids])


User 123 may like the following movies:


,title,genres
movieId,,
926,All About Eve (1950),Drama
2186,Strangers on a Train (1951),Crime|Drama|Film-Noir|Thriller
1283,High Noon (1952),Drama|Western
6188,Old School (2003),Comedy
3910,Dancer in the Dark (2000),Drama|Musical
4366,Atlantis: The Lost Empire (2001),Adventure|Animation|Children|Fantasy
5785,Jackass: The Movie (2002),Action|Comedy|Documentary
1945,On the Waterfront (1954),Crime|Drama
1680,Sliding Doors (1998),Drama|Romance
